# SIT225 5C — Smooth live Plotly Dash graphs

## 1. Setup

In [41]:
# !pip install "dash>=2.11" plotly arduino-iot-cloud

In [42]:
import time
import math
import random
import socket
import threading

import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, Patch
from dash.exceptions import PreventUpdate


def run_app(app, port=8050):
    """Show a Dash app in the notebook, using a free port if this one is busy."""
    for p in range(port, port + 20):
        with socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", p)) != 0:
                print(f"running on port {p}")
                return app.run(jupyter_mode="inline", port=p)
    print("No free port. Restart the kernel and try again.")


def start_fake_phone(send):
    """Pretend to be the phone: sends x, y and z five times a second."""
    def loop():
        t = 0
        while True:
            send({"x": math.sin(t), "y": math.sin(t + 2), "z": -1 + math.sin(t + 4)})
            time.sleep(0.2)
            t += 0.2

    threading.Thread(target=loop, daemon=True).start()


print("ready")

ready


## 2. The baseline version (the jumpy one)


In [63]:
before_data = []
start_fake_phone(before_data.append)

# a starting figure, so the cell shows something straight away
empty = go.Figure()
empty.update_layout(title="BEFORE - collecting the first 20 samples...", height=350)

before_app = Dash(__name__)
before_app.layout = html.Div([
    dcc.Graph(id="g", figure=empty),
    dcc.Interval(id="t", interval=500),
])
drawn = [0]

print("The first graph appears once 20 samples have arrived (about 4 seconds).")


@before_app.callback(Output("g", "figure"), Input("t", "n_intervals"))
def redraw(_):
    if len(before_data) - drawn[0] < 20:      # wait for 20 new samples
        raise PreventUpdate
    drawn[0] = len(before_data)

    figure = go.Figure()                      # build a whole new graph every time
    for name in ["x", "y", "z"]:
        figure.add_trace(go.Scatter(y=[p[name] for p in before_data],
                                    mode="lines", name=name))
    figure.update_layout(title="BEFORE - jumpy", height=350)
    return figure


run_app(before_app, 8050)

The first graph appears once 20 samples have arrived (about 4 seconds).
running on port 8057


## 3. The function


In [46]:
%%writefile smooth_dash.py
"""smooth_dash.py — one function that makes a Dash graph update smoothly.

SIT225 5C.

    app = Dash(__name__)
    layout, add_point = smooth_graph(app, ["x", "y", "z"], y_range=(-2, 2))
    app.layout = layout

    add_point({"x": 0.1, "y": -0.3, "z": -0.98})    # whenever data arrives

    app.run()
"""

import time
import threading

import plotly.graph_objects as go
from dash import dcc, html, Input, Output, Patch
from dash.exceptions import PreventUpdate


def smooth_graph(app, names, window=15, fps=10, y_range=None, title=""):
    """Add a smoothly-updating live graph to a Dash app.

    app     : your Dash app
    names   : list of line names, e.g. ["x", "y", "z"]
    window  : how many seconds of data to show at once
    fps     : how many times a second the graph updates
    y_range : (low, high) to fix the y axis, or None to let it scale itself
    title   : chart title

    Returns (layout, add_point).
    """
    waiting = []                  # points that have arrived but are not drawn yet
    lock = threading.Lock()       # data arrives on another thread, so we need this
    start = time.time()

    # ---- 1. the user calls this when new data arrives ----------------------
    def add_point(values):
        with lock:
            waiting.append((time.time() - start, values))

    # ---- 2. an empty graph with one line per name --------------------------
    figure = go.Figure()
    for name in names:
        figure.add_trace(go.Scatter(x=[], y=[], mode="lines", name=name))
    figure.update_layout(
        title=title,
        uirevision="keep",            # <- keeps the user's zoom when we update
        xaxis=dict(range=[0, window], title="Seconds"),
        yaxis=dict(range=list(y_range) if y_range else None),
        height=400,
    )

    layout = html.Div([
        dcc.Graph(id="smooth-graph", figure=figure),
        dcc.Interval(id="smooth-timer", interval=int(1000 / fps)),
    ])

    # ---- 3. add the waiting points to the lines ----------------------------
    # "extendData" tells Plotly to ADD to the lines already on screen, instead
    # of drawing a new graph. This is the bit that stops the flicker.
    @app.callback(Output("smooth-graph", "extendData"),
                  Input("smooth-timer", "n_intervals"))
    def draw_new_points(_):
        with lock:
            batch = list(waiting)
            waiting.clear()

        if not batch:
            raise PreventUpdate       # nothing new, do nothing

        times = [point[0] for point in batch]
        lines = []
        for name in names:
            lines.append([point[1][name] for point in batch])

        return (
            dict(x=[times] * len(names), y=lines),
            list(range(len(names))),  # which lines to add to
            1000,                     # keep the newest 1000 points
        )

    # ---- 4. slide the time window so the graph scrolls ---------------------
    # Patch() changes ONE thing (the x axis range) instead of the whole graph.
    @app.callback(Output("smooth-graph", "figure"),
                  Input("smooth-timer", "n_intervals"))
    def slide_window(_):
        left = max(0, (time.time() - start) - window)
        patch = Patch()
        patch["layout"]["xaxis"]["range"] = [left, left + window]
        return patch

    return layout, add_point

Overwriting smooth_dash.py


## 4. The AFTER version

In [48]:
import importlib
import smooth_dash
importlib.reload(smooth_dash)
from smooth_dash import smooth_graph

after_app = Dash(__name__)
layout, add_point = smooth_graph(after_app, ["x", "y", "z"],
                                 y_range=(-2, 2), title="AFTER - smooth")
after_app.layout = layout

start_fake_phone(add_point)

run_app(after_app, 8051)

running on port 8054


## 5. The real phone


In [50]:
RUN_WITH_REAL_PHONE = False        # set True when your credentials are ready

if RUN_WITH_REAL_PHONE:
    import os
    from arduino_iot_cloud import ArduinoCloudClient

    real_app = Dash(__name__)
    layout, add_point = smooth_graph(real_app, ["x", "y", "z"],
                                     y_range=(-2, 2), title="Phone accelerometer")
    real_app.layout = layout

    latest = {}

    def got_value(axis, value):
        """Collect x, y and z, then send them together as one point."""
        latest[axis] = value
        if len(latest) == 3:
            add_point(dict(latest))
            latest.clear()

    client = ArduinoCloudClient(
        device_id=os.environ["ARDUINO_DEVICE_ID"],
        username=os.environ["ARDUINO_DEVICE_ID"],
        password=os.environ["ARDUINO_SECRET_KEY"],
        sync_mode=False,
    )
    client.register("accelerometer_x", value=None, on_write=lambda c, v: got_value("x", v))
    client.register("accelerometer_y", value=None, on_write=lambda c, v: got_value("y", v))
    client.register("accelerometer_z", value=None, on_write=lambda c, v: got_value("z", v))
    threading.Thread(target=client.start, daemon=True).start()

    run_app(real_app, 8052)
else:
    print("Set RUN_WITH_REAL_PHONE = True")

Set RUN_WITH_REAL_PHONE = True


## 6. The same function with different data


In [52]:
other_app = Dash(__name__)
layout, add_other = smooth_graph(other_app, ["temperature", "humidity"],
                                 window=20, y_range=(0, 100),
                                 title="Same function, different data")
other_app.layout = layout


def fake_sensor():
    temp, hum = 22.0, 55.0
    while True:
        temp = min(35, max(15, temp + random.gauss(0, 0.4)))
        hum = min(90, max(30, hum + random.gauss(0, 1.0)))
        add_other({"temperature": temp, "humidity": hum})
        time.sleep(0.5)


threading.Thread(target=fake_sensor, daemon=True).start()

run_app(other_app, 8053)

running on port 8055


## 7. A data file for the repo


In [54]:
import csv

rows = []
start_fake_phone(lambda v: rows.append(v))
time.sleep(20)

with open("accelerometer_5c.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["x", "y", "z"])
    for r in rows:
        writer.writerow([r["x"], r["y"], r["z"]])

print(f"saved {len(rows)} rows to accelerometer_5c.csv")

saved 198 rows to accelerometer_5c.csv
